<a href="https://colab.research.google.com/github/tanpaterusan/data-science-2026/blob/main/Pertemuan3_Amalia_240401010284.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# PIPELINE DATA CLEANING — HOUSING DATASET
# ============================================
import pandas as pd, numpy as np
from scipy.stats.mstats import winsorize


In [ ]:
# STEP 0 — Load & eksplorasi awal

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Data Science/housing_dirty.csv')
print('Shape awal:', df.shape)
print(df.isnull().sum())

print(df.info)
print(df.describe())

df

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape awal: (130, 7)
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64
<bound method DataFrame.info of       id  luas_m2  harga_juta        kota  kamar  tahun_bangun kondisi
0      1    297.0      1084.0       jogja    2.0          2000    baik
1      2    254.0       761.0       Medan    NaN          1995   Bagus
2      3    249.7       895.0       Depok    NaN          1983    baik
3      4     49.7       178.0         YGY    5.0          2013    baik
4      5    133.4       424.0       Medan    5.0          2004  Sedang
..   ...      ...         ...         ...    ...           ...     ...
125  126    330.2      1050.0    Semarang    6.0          1993    Baik
126  127      NaN         NaN    Makassar    5.0          2008    baik
127  128     84.1       202.0    

,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang
...,...,...,...,...,...,...,...
125,126,330.2,1050.0,Semarang,6.0,1993,Baik
126,127,NaN,NaN,Makassar,5.0,2008,baik
127,128,84.1,202.0,Makassar,2.0,1986,Baik
128,129,88.0,252.0,makassar,NaN,1986,Bagus


In [ ]:
# STEP 1 — Hapus Duplikat
df.drop_duplicates(inplace=True)
print('Setelah hapus duplikat:', df.shape)

df



Setelah hapus duplikat: (130, 7)


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang
...,...,...,...,...,...,...,...
125,126,330.2,1050.0,Semarang,6.0,1993,Baik
126,127,NaN,NaN,Makassar,5.0,2008,baik
127,128,84.1,202.0,Makassar,2.0,1986,Baik
128,129,88.0,252.0,makassar,NaN,1986,Bagus


In [ ]:
# STEP 2 — Normalisasi String
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

df

,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,Jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,Ygy,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,sedang
...,...,...,...,...,...,...,...
125,126,330.2,1050.0,Semarang,6.0,1993,baik
126,127,NaN,NaN,Makassar,5.0,2008,baik
127,128,84.1,202.0,Makassar,2.0,1986,baik
128,129,88.0,252.0,Makassar,NaN,1986,bagus


In [ ]:
# STEP 3 — Imputasi Missing Values
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

df

,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,Jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,1.0,1995,bagus
2,3,249.7,895.0,Depok,1.0,1983,baik
3,4,49.7,178.0,Ygy,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,sedang
...,...,...,...,...,...,...,...
125,126,330.2,1050.0,Semarang,6.0,1993,baik
126,127,193.8,655.0,Makassar,5.0,2008,baik
127,128,84.1,202.0,Makassar,2.0,1986,baik
128,129,88.0,252.0,Makassar,1.0,1986,bagus


In [ ]:
# STEP 4 — Tangani Outlier (IQR Fence)
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
  Q1, Q3 = df[col].quantile([0.25, 0.75])

IQR = Q3-Q1
df[col] = df[col].clip(Q1-1.5*IQR, Q3+1.5*IQR)

df


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,Jogja,2.0,2000.0,baik
1,2,254.0,761.0,Medan,1.0,1995.0,bagus
2,3,249.7,895.0,Depok,1.0,1983.0,baik
3,4,49.7,178.0,Ygy,5.0,2013.0,baik
4,5,133.4,424.0,Medan,5.0,2004.0,sedang
...,...,...,...,...,...,...,...
125,126,330.2,1050.0,Semarang,6.0,1993.0,baik
126,127,193.8,655.0,Makassar,5.0,2008.0,baik
127,128,84.1,202.0,Makassar,2.0,1986.0,baik
128,129,88.0,252.0,Makassar,1.0,1986.0,bagus


In [ ]:
# STEP 5 — Validasi & Ekspor
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'
print('Shape akhir:', df.shape)
df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih tersimpan!')

Shape akhir: (130, 7)
Dataset bersih tersimpan!


**Evaluasi**

1. Apa yang Saya Pelajari
- Pipeline Cleaning: Urutan membersihkan data kotor dari awal sampai siap pakai.
- Standardisasi Teks: Menghapus spasi liar dan menyamakan huruf kapital/kecil pada data teks.
- Imputasi Data: Mengisi kolom kosong memakai nilai Tengah (Median) untuk angka dan Modus untuk kategori.
- Pagar IQR: Memotong nilai ekstrem (outlier) agar masuk ke rentang yang masuk akal.
- Validasi Otomatis: Memakai perintah assert untuk menjamin tidak ada lagi data kosong/duplikat sebelum disimpan.


---



2. Temuan Utama Data:
- Data Sangat Kotor: Banyak nilai kosong di kolom luas, harga, dan jumlah kamar.
- Anomali Mustahil: Ditemukan luas dan harga bernilai negatif, serta tahun bangun rumah tertulis angka 9999.
- Outlier Ekstrem: Ada rumah dengan harga 100 juta dan luas 9500 $m^2$ yang merusak nilai rata-rata.
- Ejaan Berantakan: Nama kota ditulis tidak seragam (contoh: "jogja", "YGY", "yogyakarta"), padahal maksudnya sama.


---



3. Keterbatasan & Pertanyaan Belajar:
- Bug Code (Fatal): Ada salah ketik indentasi di Langkah 4.
- Fungsi .clip() berada di luar perulangan, sehingga hanya kolom tahun yang dibersihkan, sedangkan harga dan luas masih rusak.
- Salah Urutan: Mengapa mengisi data kosong (imputasi) dilakukan sebelum membuang data rusak (outlier)? Ini bisa membuat nilai pengisinya bias.
- Kota Belum Sinkron: Fungsi .title() hanya merapikan huruf, tapi "Ygy" dan "Yogyakarta" tetap dianggap dua kota berbeda oleh sistem. Mengapa tidak disatukan?
- Imputasi Kurang Akurat: Rumah besar otomatis diberi 1 kamar hanya karena angka 1 adalah modus (paling sering muncul). Apakah tidak sebaiknya dikelompokkan dulu berdasarkan luasnya?